In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
pd.set_option('display.max_rows', None)

In [54]:
X_train = pd.read_csv('csv/final_datasets/for_linear_model/DF/predictors_train_filtered.csv')
X_test = pd.read_csv('csv/final_datasets/for_linear_model/DF/predictors_test_filtered.csv')
y_train = pd.read_csv('csv/final_datasets/for_linear_model/DF/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('csv/final_datasets/for_linear_model/DF/target_test_filtered_logariphmed.csv')

In [5]:
train_players = pd.read_csv('csv/final_datasets/for_linear_model/DF/train_players_names.csv')
test_players = pd.read_csv('csv/final_datasets/for_linear_model/DF/test_players_names.csv')

In [6]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

## Регрессор

In [56]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

## Предсказанные значения

In [57]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

## Метрики

In [58]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

## Остатки

In [59]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

## Коэффициенты

In [60]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

# Все переменные

In [99]:
coeffs_df

,coeff
90s,23.536915
Min,-23.505636
G+A.1,-1.039924
G+A-PK,1.027266
league_GB1,0.644170
Age,-0.574810
G-PK.1,-0.368688
league_IT1,0.364808
league_L1,0.324781
Starts,0.314362


In [13]:
metrics

,MAE,RMSE,MAPE,R2
test,5.741616,12.221873,0.904190,0.184838
train,4.626652,9.346249,0.872897,0.382257


In [17]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
664,Milos Kerkez,95.261310,45.000
730,Nathan Collins,74.871969,28.000
412,Dean Huijsen,66.580929,60.000
784,Daniel Munoz,62.514942,25.000
890,Levi Colwill,58.217044,55.000
56,Trent Alexander-Arnold,54.636312,75.000
342,Nikola Milenkovic,54.625294,35.000
25,Neco Williams,45.419707,20.000
10,Pedro Porro,44.669265,38.000
472,Jurrien Timber,41.614041,55.000


In [16]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
23,Rayan Ait-Nouri,139.935654,35.000
77,Josko Gvardiol,65.011485,75.000
168,Antonee Robinson,46.375016,35.000
179,Marc Cucurella,34.725967,35.000
136,Gabriel Magalhaes,31.674999,75.000
86,Ibrahima Konate,31.669813,60.000
82,Calvin Bassey,28.116200,25.000
201,Marc Guehi,27.954755,45.000
60,Ian Maatsen,27.933642,28.000
227,Myles Lewis-Skelly,27.568274,45.000


In [27]:
comp_leftovers_train[comp_leftovers_train['Player'].str.contains('Petrov')]

,Player,Value_pred,Value
288,Hristiyan Petrov,0.911652,0.6
306,Sergei Petrov,0.493325,0.4


# Потенциальные переменные для добавления и удаления

In [158]:
base_cols = ['G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

In [160]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,5.372967,0.917254,0.405040
MP,5.372435,1.135546,-0.254711
Starts,5.372608,1.131070,-0.254962
Min,5.346980,1.134117,-0.369775
Gls,5.370874,1.137968,-0.250175
PK,5.370874,1.137968,-0.250175
PKatt,5.372805,1.137722,-0.246176
CrdR,5.371681,1.141755,-0.248972
Fls,5.367723,1.150526,-0.362269
Off,5.371714,1.139739,-0.252983


In [161]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,5.372967,0.917254,0.405040
Int,5.371651,1.140412,-0.272437
G/Sh,5.369203,1.139141,-0.269325


# Модель 1

In [31]:
X_train = X_train[['G+A', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'Fld', 'Fls', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'Fld', 'Fls', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [37]:
coeffs_df

,coeff
league_GB1,0.644750
Age,-0.586466
90s,0.363879
league_IT1,0.363689
league_L1,0.323772
league_ES1,0.299015
Ast,0.288419
league_FR1,0.271033
G+A,-0.160010
G-PK,0.159311


In [38]:
metrics

,MAE,RMSE,MAPE,R2
test,5.669392,11.283822,0.930519,0.305166
train,4.679034,9.559381,0.886455,0.353762


In [45]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
664,Milos Kerkez,110.517417,45.000
412,Dean Huijsen,69.441752,60.000
10,Pedro Porro,68.833300,38.000
56,Trent Alexander-Arnold,62.048535,75.000
730,Nathan Collins,59.365208,28.000
890,Levi Colwill,56.788238,55.000
342,Nikola Milenkovic,49.625804,35.000
503,Murillo,44.277455,55.000
784,Daniel Munoz,44.093253,25.000
178,Lewis Hall,41.802382,32.000


In [46]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
23,Rayan Ait-Nouri,104.374223,35.000
77,Josko Gvardiol,67.675446,75.000
168,Antonee Robinson,53.727674,35.000
201,Marc Guehi,36.639039,45.000
179,Marc Cucurella,31.037772,35.000
60,Ian Maatsen,27.540945,28.000
227,Myles Lewis-Skelly,27.308639,45.000
86,Ibrahima Konate,27.150344,60.000
136,Gabriel Magalhaes,25.938437,75.000
110,Vitaliy Mykolenko,25.143597,28.000


# Модель 2

In [51]:
X_train = X_train[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'Fls', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'Fls', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [57]:
coeffs_df

,coeff
league_GB1,0.640705
Age,-0.585580
90s,0.364311
league_IT1,0.357902
league_L1,0.323469
league_ES1,0.296088
league_FR1,0.265908
Ast,0.245730
G-PK,0.189217
TklW,-0.138398


In [58]:
metrics

,MAE,RMSE,MAPE,R2
test,5.613919,11.363877,0.920371,0.295272
train,4.663112,9.586427,0.882595,0.350100


In [59]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
664,Milos Kerkez,115.716721,45.000
412,Dean Huijsen,68.397480,60.000
10,Pedro Porro,66.809384,38.000
730,Nathan Collins,61.832988,28.000
56,Trent Alexander-Arnold,58.321354,75.000
890,Levi Colwill,56.880083,55.000
342,Nikola Milenkovic,51.616052,35.000
784,Daniel Munoz,44.903736,25.000
503,Murillo,42.753415,55.000
178,Lewis Hall,40.762975,32.000


In [60]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
23,Rayan Ait-Nouri,110.516296,35.000
77,Josko Gvardiol,71.804358,75.000
168,Antonee Robinson,49.664445,35.000
201,Marc Guehi,39.058889,45.000
179,Marc Cucurella,34.969210,35.000
227,Myles Lewis-Skelly,28.647967,45.000
60,Ian Maatsen,27.739204,28.000
136,Gabriel Magalhaes,26.596850,75.000
86,Ibrahima Konate,26.559494,60.000
110,Vitaliy Mykolenko,25.248342,28.000


# Модель 3

In [66]:
X_train = X_train[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', 'Sh', 'SoT/90', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [72]:
coeffs_df

,coeff
league_GB1,0.633083
Age,-0.588033
90s,0.398699
league_IT1,0.361135
league_L1,0.319165
league_ES1,0.298034
league_FR1,0.265436
Ast,0.241660
G-PK,0.193043
league_NL1,-0.141574


In [73]:
metrics

,MAE,RMSE,MAPE,R2
test,5.600638,11.405773,0.911615,0.290066
train,4.641290,9.544416,0.886544,0.355784


In [74]:
comp_leftovers_train

,Player,Value_pred,Value
0,Alessandro Marcandalli,4.824710,2.000
1,Tyrick Mitchell,34.312494,25.000
2,Valentin Paltsev,3.272173,2.500
3,Teo Quintero,0.703826,0.600
4,Kaio Pantaleao,0.748273,1.800
5,Aitor Paredes,6.178441,22.000
6,Fredrik Oppegard,1.463485,0.600
7,Sergei Bozhin,1.042034,0.700
8,Francisco Petrasso,1.270452,1.500
9,Dara O'Shea,30.593307,14.000


In [75]:
comp_leftovers_test

,Player,Value_pred,Value
0,Jon Aramburu,7.382454,15.000
1,Alphonso Davies,9.921329,50.000
2,Riccardo Calafiori,18.451519,35.000
3,Fali Cande,4.265522,2.500
4,Romain Perraud,7.839115,5.000
5,Harry Clarke,8.403590,4.500
6,Boyd Reith,0.663792,0.400
7,Zineddine Belaid,1.304509,0.800
8,Yerry Mina,5.391975,3.000
9,Aiham Ousou,2.256036,2.500


# Модель 4

In [55]:
X_train = X_train[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Ast', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [62]:
coeffs_df

,coeff
league_GB1,0.626925
Age,-0.588286
90s,0.431176
league_IT1,0.352984
league_L1,0.312027
league_ES1,0.291146
league_FR1,0.257755
Ast,0.213310
G-PK,0.185899
Sh/90,0.166882


In [61]:
metrics

,MAE,RMSE,MAPE,R2
test,5.566665,11.325218,0.913154,0.300058
train,4.647761,9.568882,0.886695,0.352477


In [63]:
comp_leftovers_train

,Player,Value_pred,Value
0,Alessandro Marcandalli,4.732267,2.000
1,Tyrick Mitchell,35.508587,25.000
2,Valentin Paltsev,3.064061,2.500
3,Teo Quintero,0.714488,0.600
4,Kaio Pantaleao,0.747734,1.800
5,Aitor Paredes,6.375678,22.000
6,Fredrik Oppegard,1.489391,0.600
7,Sergei Bozhin,1.026532,0.700
8,Francisco Petrasso,1.282803,1.500
9,Dara O'Shea,30.227170,14.000


In [64]:
comp_leftovers_test

,Player,Value_pred,Value
0,Jon Aramburu,7.562272,15.000
1,Alphonso Davies,9.947059,50.000
2,Riccardo Calafiori,18.211865,35.000
3,Fali Cande,4.290052,2.500
4,Romain Perraud,7.862222,5.000
5,Harry Clarke,8.099773,4.500
6,Boyd Reith,0.648006,0.400
7,Zineddine Belaid,1.324183,0.800
8,Yerry Mina,5.116593,3.000
9,Aiham Ousou,2.272183,2.500


# Модель 5

In [44]:
X_train = X_train[['G+A', 'G+A.1', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [50]:
coeffs_df

,coeff
league_GB1,0.629187
Age,-0.589858
90s,0.431591
league_IT1,0.355601
league_L1,0.314257
league_ES1,0.291664
league_FR1,0.258124
G+A,0.207486
Sh/90,0.166010
G+A.1,-0.150654


In [51]:
metrics

,MAE,RMSE,MAPE,R2
test,5.550672,11.212241,0.914493,0.313954
train,4.637658,9.532799,0.886727,0.357351


In [52]:
comp_leftovers_train

,Player,Value_pred,Value
0,Alessandro Marcandalli,4.760061,2.000
1,Tyrick Mitchell,34.880581,25.000
2,Valentin Paltsev,3.063554,2.500
3,Teo Quintero,0.706990,0.600
4,Kaio Pantaleao,0.738747,1.800
5,Aitor Paredes,6.324494,22.000
6,Fredrik Oppegard,1.490828,0.600
7,Sergei Bozhin,1.019418,0.700
8,Francisco Petrasso,1.258889,1.500
9,Dara O'Shea,30.011124,14.000


In [53]:
comp_leftovers_test

,Player,Value_pred,Value
0,Jon Aramburu,7.632123,15.000
1,Alphonso Davies,9.927970,50.000
2,Riccardo Calafiori,18.151490,35.000
3,Fali Cande,4.293745,2.500
4,Romain Perraud,7.744173,5.000
5,Harry Clarke,8.129064,4.500
6,Boyd Reith,0.651621,0.400
7,Zineddine Belaid,1.318612,0.800
8,Yerry Mina,5.139381,3.000
9,Aiham Ousou,2.264139,2.500


# Модель 6

In [33]:
X_train = X_train[['G+A.1', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A.1', 'G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [39]:
coeffs_df

,coeff
league_GB1,0.637359
Age,-0.584323
90s,0.472828
league_IT1,0.366363
league_L1,0.323853
league_ES1,0.297940
league_FR1,0.266634
Ast.1,0.236942
Sh/90,0.171286
G-PK,0.161150


In [40]:
metrics

,MAE,RMSE,MAPE,R2
test,5.447071,10.546331,0.900855,0.393024
train,4.585201,9.367524,0.893122,0.379442


In [41]:
comp_leftovers_train

,Player,Value_pred,Value
0,Alessandro Marcandalli,4.580892,2.000
1,Tyrick Mitchell,29.126872,25.000
2,Valentin Paltsev,3.353092,2.500
3,Teo Quintero,0.699054,0.600
4,Kaio Pantaleao,0.732866,1.800
5,Aitor Paredes,6.471008,22.000
6,Fredrik Oppegard,1.460603,0.600
7,Sergei Bozhin,1.054437,0.700
8,Francisco Petrasso,1.189564,1.500
9,Dara O'Shea,29.122953,14.000


In [42]:
comp_leftovers_test

,Player,Value_pred,Value
0,Jon Aramburu,8.343553,15.000
1,Alphonso Davies,9.824185,50.000
2,Riccardo Calafiori,18.613235,35.000
3,Fali Cande,4.322374,2.500
4,Romain Perraud,7.395978,5.000
5,Harry Clarke,7.697380,4.500
6,Boyd Reith,0.662487,0.400
7,Zineddine Belaid,1.261718,0.800
8,Yerry Mina,5.694794,3.000
9,Aiham Ousou,2.266363,2.500


# Модель 7

In [21]:
X_train = X_train[['G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [27]:
coeffs_df

,coeff
league_GB1,0.641083
Age,-0.584654
90s,0.498375
league_IT1,0.369875
league_L1,0.323487
league_ES1,0.299119
league_FR1,0.268603
Sh/90,0.153379
league_NL1,-0.135465
Ast.1,0.123801


In [28]:
metrics

,MAE,RMSE,MAPE,R2
test,5.372967,10.441416,0.917254,0.405040
train,4.588574,9.361336,0.897543,0.380261


In [29]:
comp_leftovers_train

,Player,Value_pred,Value
0,Alessandro Marcandalli,4.432573,2.000
1,Tyrick Mitchell,31.069978,25.000
2,Valentin Paltsev,3.358897,2.500
3,Teo Quintero,0.774306,0.600
4,Kaio Pantaleao,0.751695,1.800
5,Aitor Paredes,6.496157,22.000
6,Fredrik Oppegard,1.366992,0.600
7,Sergei Bozhin,1.043633,0.700
8,Francisco Petrasso,1.418022,1.500
9,Dara O'Shea,30.554910,14.000


In [30]:
comp_leftovers_test

,Player,Value_pred,Value
0,Jon Aramburu,8.417943,15.000
1,Alphonso Davies,9.521763,50.000
2,Riccardo Calafiori,20.263305,35.000
3,Fali Cande,4.427993,2.500
4,Romain Perraud,7.272842,5.000
5,Harry Clarke,7.531020,4.500
6,Boyd Reith,0.649284,0.400
7,Zineddine Belaid,1.218045,0.800
8,Yerry Mina,5.702930,3.000
9,Aiham Ousou,2.275199,2.500


# Модель 8

In [7]:
X_train = X_train[['G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G-PK', 'Ast.1', '90s', 'CrdY', 'Sh/90', 'TklW', 'Age', 'Int', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [179]:
coeffs_df

,coeff
league_GB1,0.641447
Age,-0.584517
90s,0.499064
league_IT1,0.370106
league_L1,0.323713
league_ES1,0.299498
league_FR1,0.268970
Sh/90,0.154092
league_NL1,-0.135165
Ast.1,0.123980


In [180]:
metrics

,MAE,RMSE,MAPE,R2
test,5.369203,10.435910,0.917497,0.405668
train,4.587877,9.360052,0.897648,0.380431


In [182]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
664,Milos Kerkez,87.317019,45.000
412,Dean Huijsen,61.453324,60.000
730,Nathan Collins,56.144334,28.000
890,Levi Colwill,55.663988,55.000
503,Murillo,50.295091,55.000
342,Nikola Milenkovic,49.465111,35.000
10,Pedro Porro,47.956962,38.000
56,Trent Alexander-Arnold,46.491784,75.000
206,Illia Zabarnyi,42.492948,42.000
178,Lewis Hall,41.452132,32.000


In [183]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
77,Josko Gvardiol,80.247596,75.000
23,Rayan Ait-Nouri,72.037168,35.000
201,Marc Guehi,40.093512,45.000
179,Marc Cucurella,34.408033,35.000
82,Calvin Bassey,30.894256,25.000
227,Myles Lewis-Skelly,30.557671,45.000
168,Antonee Robinson,29.550028,35.000
60,Ian Maatsen,28.775205,28.000
110,Vitaliy Mykolenko,27.779017,28.000
136,Gabriel Magalhaes,26.142407,75.000


In [19]:
comp_leftovers_train[comp_leftovers_train['Player'].str.contains('Diego')]

,Player,Value_pred,Value
577,Diego Llorente,3.908016,6.0
821,Diego Carlos,3.579433,9.5
880,Diego,2.552908,8.0
